# 10 — TXT to CSV Conversion

Converts all TXT files in `Houston_data/` to CSV format.

**Process:**
1. Scan `Houston_data/` for all `.txt` files (recursive)
2. Detect delimiter (tab, comma, pipe, space)
3. Read with pandas
4. Save as CSV to `Houston_data/csv_files/`

**Output:** CSV files in `Houston_data/csv_files/` with same names as originals

In [1]:
import pandas as pd
import pathlib
import os

# ── Setup paths ───────────────────────────────────────
HOUSTON_DATA_DIR = "Houston_data"
CSV_OUTPUT_DIR = f"{HOUSTON_DATA_DIR}/csv_files"

# Create output directory if it doesn't exist
os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

print(f"Input folder:  {HOUSTON_DATA_DIR}/")
print(f"Output folder: {CSV_OUTPUT_DIR}/")
print()

Input folder:  Houston_data/
Output folder: Houston_data/csv_files/



In [2]:
# ── Find all TXT files ────────────────────────────────
txt_files = list(pathlib.Path(HOUSTON_DATA_DIR).rglob("*.txt"))
print(f"Found {len(txt_files)} TXT files:")
for f in txt_files:
    print(f"  {f.relative_to(pathlib.Path('.'))}")
print()

Found 50 TXT files:
  Houston_data\Code_description_real\desc_r_01_state_class.txt
  Houston_data\Code_description_real\desc_r_02_building_type_code.txt
  Houston_data\Code_description_real\desc_r_03_building_style.txt
  Houston_data\Code_description_real\desc_r_04_building_class.txt
  Houston_data\Code_description_real\desc_r_05_building_data_elements.txt
  Houston_data\Code_description_real\desc_r_06_structural_element_type.txt
  Houston_data\Code_description_real\desc_r_07_quality_code.txt
  Houston_data\Code_description_real\desc_r_08_pgi_category.txt
  Houston_data\Code_description_real\desc_r_09_subarea_type.txt
  Houston_data\Code_description_real\desc_r_10_extra_features.txt
  Houston_data\Code_description_real\desc_r_11_extra_feature_category.txt
  Houston_data\Code_description_real\desc_r_12_real_jurisdictions.txt
  Houston_data\Code_description_real\desc_r_13_real_jurisdiction_type.txt
  Houston_data\Code_description_real\desc_r_14_exemption_category.txt
  Houston_data\Code_

In [3]:
# ── Helper: detect delimiter ──────────────────────────
def detect_delimiter(file_path, sample_lines=5):
    """
    Auto-detect delimiter by checking first few lines.
    Tries: tab, comma, pipe, space (in that order).
    """
    delimiters = ["\t", ",", "|", " "]
    
    with open(file_path, "r", encoding="utf-8") as f:
        sample = [next(f) for _ in range(sample_lines)]
    
    for delim in delimiters:
        # Check if all sample lines have consistent column count with this delimiter
        col_counts = [len(line.split(delim)) for line in sample]
        if len(set(col_counts)) == 1 and col_counts[0] > 1:
            return delim
    
    return "\t"  # Default to tab

print("Delimiter detection helper ready.")

Delimiter detection helper ready.


In [4]:
# ── Convert TXT files to CSV ──────────────────────────
results = {}

for txt_file in txt_files:
    try:
        # Detect delimiter
        delim = detect_delimiter(txt_file)
        delim_name = {"\t": "TAB", ",": "COMMA", "|": "PIPE", " ": "SPACE"}[delim]
        
        # Read TXT file
        df = pd.read_csv(txt_file, delimiter=delim, dtype=str)
        
        # Generate output path (preserve subdirectory structure)
        rel_path = txt_file.relative_to(HOUSTON_DATA_DIR)
        csv_output_path = pathlib.Path(CSV_OUTPUT_DIR) / rel_path.with_suffix(".csv")
        csv_output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Save as CSV
        df.to_csv(csv_output_path, index=False)
        
        results[str(rel_path)] = {
            "status": "OK",
            "delimiter": delim_name,
            "rows": len(df),
            "cols": df.shape[1],
            "output": str(csv_output_path),
        }
        
        print(f"✓ {rel_path}")
        print(f"  Delimiter: {delim_name} | {len(df)} rows x {df.shape[1]} cols")
        print(f"  → {csv_output_path}")
        print()
        
    except Exception as e:
        results[str(rel_path)] = {"status": "FAILED", "error": str(e)}
        print(f"✗ {rel_path}")
        print(f"  Error: {e}")
        print()

✓ Code_description_real\desc_r_01_state_class.txt
  Delimiter: TAB | 61 rows x 3 cols
  → Houston_data\csv_files\Code_description_real\desc_r_01_state_class.csv

✓ Code_description_real\desc_r_02_building_type_code.txt
  Delimiter: TAB | 196 rows x 2 cols
  → Houston_data\csv_files\Code_description_real\desc_r_02_building_type_code.csv

✓ Code_description_real\desc_r_03_building_style.txt
  Delimiter: TAB | 421 rows x 2 cols
  → Houston_data\csv_files\Code_description_real\desc_r_03_building_style.csv

✓ Code_description_real\desc_r_04_building_class.txt
  Delimiter: TAB | 9 rows x 2 cols
  → Houston_data\csv_files\Code_description_real\desc_r_04_building_class.csv

✓ Code_description_real\desc_r_05_building_data_elements.txt
  Delimiter: TAB | 108 rows x 3 cols
  → Houston_data\csv_files\Code_description_real\desc_r_05_building_data_elements.csv

✓ Code_description_real\desc_r_06_structural_element_type.txt
  Delimiter: TAB | 190 rows x 3 cols
  → Houston_data\csv_files\Code_descripti

In [5]:
# ── Summary ───────────────────────────────────────────
print("\n" + "="*60)
print("  CONVERSION SUMMARY")
print("="*60)

successful = [r for r in results.values() if r["status"] == "OK"]
failed = [r for r in results.values() if r["status"] == "FAILED"]

print(f"\nTotal files: {len(results)}")
print(f"Successful:  {len(successful)}")
print(f"Failed:      {len(failed)}")

if successful:
    print(f"\nCSV files saved to: {CSV_OUTPUT_DIR}/")
    total_rows = sum(r["rows"] for r in successful)
    print(f"Total rows combined: {total_rows:,}")

if failed:
    print("\nFailed conversions:")
    for name, result in results.items():
        if result["status"] == "FAILED":
            print(f"  {name}: {result['error']}")

print()


  CONVERSION SUMMARY

Total files: 44
Successful:  41
Failed:      3

CSV files saved to: Houston_data/csv_files/
Total rows combined: 59,508,240

Failed conversions:
  Real_acct_owner\deeds.txt: 'utf-8' codec can't decode byte 0xa5 in position 235078: invalid start byte
  Real_acct_owner\parcel_tieback.txt: 'utf-8' codec can't decode byte 0xa5 in position 227599: invalid start byte
  Real_acct_owner\real_neighborhood_code.txt: 'utf-8' codec can't decode byte 0xff in position 699: invalid start byte



In [6]:
import pandas as pd

# Read first 5 rows to see columns
df_land = pd.read_csv("Houston_data/csv_files/Real_building_land/land.csv", nrows=5)
print("land.csv columns:", df_land.columns.tolist())
print("land.csv shape:", df_land.shape)

df_struct = pd.read_csv("Houston_data/csv_files/Real_building_land/structural_elem1.csv", nrows=5)
print("\nstructural_elem1.csv columns:", df_struct.columns.tolist())
print("structural_elem1.csv shape:", df_struct.shape)

# Look for join key
print("\nland.csv head:\n", df_land.head())
print("\nstructural_elem1.csv head:\n", df_struct.head())

land.csv columns: ['acct', 'num', 'use_cd', 'use_dscr', 'inf_cd', 'inf_dscr', 'inf_adj', 'tp', 'uts', 'sz_fact', 'inf_fact', 'cond', 'ovr_dscr', 'tot_adj', 'unit_prc', 'adj_unit_prc', 'val', 'ovr_val']
land.csv shape: (5, 18)

structural_elem1.csv columns: ['acct', 'bld_num', 'code', 'adj', 'type', 'type_dscr', 'category_dscr', 'dor_cd']
structural_elem1.csv shape: (5, 8)

land.csv head:
           acct  num  use_cd                     use_dscr  inf_cd inf_dscr  \
0  10010000013    1    8005  Land Neighborhood Section 5    4600            
1  10010000013    2    8005  Land Neighborhood Section 5    4600            
2  10010000013    3    8005  Land Neighborhood Section 5    4600            
3  10020000001    1    8005  Land Neighborhood Section 5    4339            
4  10020000003    1    7001            UDI Improved Land    4339            

   inf_adj  tp      uts  sz_fact  inf_fact  cond                  ovr_dscr  \
0      1.0  SF   9775.0      1.0       1.0  0.65  Previous Flooding

In [7]:
import pandas as pd

# Check structural_elem2.csv
df_struct2 = pd.read_csv("Houston_data/csv_files/Real_building_land/structural_elem2.csv", nrows=5)
print("structural_elem2.csv columns:", df_struct2.columns.tolist())
print(df_struct2.head())

# Check for coordinates
for fname in ["extra_features.csv", "fixtures.csv", "exterior.csv", "land_detail.csv"]:
    try:
        df = pd.read_csv(f"Houston_data/csv_files/Real_building_land/{fname}", nrows=5)
        print(f"\n{fname} columns:", df.columns.tolist())
    except:
        print(f"\n{fname} not found or error reading")

# Check if coordinates are in account/parcel data
df_tieback = pd.read_csv("Houston_data/csv_files/Real_acct_owner/parcel_tieback.csv", nrows=5)
print("\nparcel_tieback.csv columns:", df_tieback.columns.tolist())

structural_elem2.csv columns: ['acct', 'bld_num', 'code', 'adj', 'type', 'type_dscr', 'category_dscr', 'dor_cd']
          acct  bld_num  code  adj  type           type_dscr category_dscr  \
0  10020000016        1     1  0.0  HTG         Heating Type       Hot Air   
1  10020000016        1     2  0.0  PAR       Partition Type        Normal   
2  10020000016        1     4  1.0  PC    Physical Condition          Good   
3  10020000016        1     2  0.0  PLM        Plumbing Type      Adequate   
4  10020000016        1     0  0.0  SPR       Sprinkler Type           NaN   

  dor_cd  
0   F1    
1   F1    
2   F1    
3   F1    
4   F1    

extra_features.csv columns: ['acct', 'bld_num', 'count', 'grade', 'cd', 's_dscr', 'l_dscr', 'cat', 'dscr', 'note', 'uts']

fixtures.csv columns: ['acct', 'bld_num', 'type', 'type_dscr', 'units']

exterior.csv columns: ['acct', 'bld_num', 'sar_cd', 'sar_dscr', 'area']

land_detail.csv not found or error reading

parcel_tieback.csv columns: ['acct', '

In [8]:
import pandas as pd
import os

# Scan ALL CSVs in Real_building_land for lat/lon/year/floor columns
building_land_dir = "Houston_data/csv_files/Real_building_land"

print("Searching for coordinates, year built, and floor information...\n")

for csv_file in sorted(os.listdir(building_land_dir)):
    if csv_file.endswith(".csv"):
        try:
            df = pd.read_csv(f"{building_land_dir}/{csv_file}", nrows=5)
            cols = df.columns.tolist()
            
            # Look for lat/lon/coordinates
            has_coords = any(x.lower() in str(cols).lower() for x in ["lat", "lon", "geo", "coord", "x ", "y ", "north", "east"])
            # Look for year
            has_year = any(x.lower() in str(cols).lower() for x in ["year", "age", "built", "construct", "vintage"])
            # Look for floors/stories
            has_floors = any(x.lower() in str(cols).lower() for x in ["floor", "story", "stry", "level", "height"])
            
            if has_coords or has_year or has_floors:
                print(f"✓ {csv_file}")
                print(f"  Columns: {cols}")
                print()
        except Exception as e:
            pass

print("\n" + "="*60)
print("Also check the first 5 rows of these files for data patterns:\n")

# Show extra_features and exterior data to see if they contain building area
for fname in ["extra_features.csv", "exterior.csv"]:
    try:
        df = pd.read_csv(f"{building_land_dir}/{fname}", nrows=5)
        print(f"\n{fname}:")
        print(df)
    except:
        pass

Searching for coordinates, year built, and floor information...


Also check the first 5 rows of these files for data patterns:


extra_features.csv:
          acct  bld_num  count  grade      cd   s_dscr                l_dscr  \
0  10020000001        0      1      4  CPA1     PavAsp      Paving - Asphalt   
1  10020000013        0      1      4  CPA1     PavAsp      Paving - Asphalt   
2  10020000015        0      1      4  CPA1     PavAsp      Paving - Asphalt   
3  10020000016        1      1      4  CCP6     CpRfSl  CANOPY ROOF AND SLAB   
4  10040000001        1      5      4  CEN5    Enclos5    Enclosure,  Retail   

      cat           dscr          note     uts  
0  MS      Miscellaneous  NEW FOR 2018  5000.0  
1  MS      Miscellaneous                3000.0  
2  MS      Miscellaneous                1250.0  
3  MS      Miscellaneous                1504.0  
4  MS      Miscellaneous                5880.0  

exterior.csv:
          acct  bld_num  sar_cd              sar_dscr   area

In [9]:
import pandas as pd
import os

# Check ALL remaining files in Real_building_land
building_land_dir = "Houston_data/csv_files/Real_building_land"

print("ALL files in Real_building_land:\n")
for csv_file in sorted(os.listdir(building_land_dir)):
    if csv_file.endswith(".csv"):
        try:
            df = pd.read_csv(f"{building_land_dir}/{csv_file}", nrows=1)
            print(f"{csv_file}: {df.columns.tolist()}")
        except Exception as e:
            print(f"{csv_file}: ERROR - {e}")

ALL files in Real_building_land:

exterior.csv: ['acct', 'bld_num', 'sar_cd', 'sar_dscr', 'area']
extra_features.csv: ['acct', 'bld_num', 'count', 'grade', 'cd', 's_dscr', 'l_dscr', 'cat', 'dscr', 'note', 'uts']
extra_features_detail1.csv: ['acct', 'cd', 'dscr', 'grade', 'cond_cd', 'bld_num', 'length', 'width', 'units', 'unit_price', 'adj_unit_price', 'pct_comp', 'act_yr', 'eff_yr', 'roll_yr', 'DT', 'pct_cond', 'dpr_val', 'note', 'asd_val']
extra_features_detail2.csv: ['acct', 'cd', 'dscr', 'grade', 'cond_cd', 'bld_num', 'length', 'width', 'units', 'unit_price', 'adj_unit_price', 'pct_comp', 'act_yr', 'eff_yr', 'roll_yr', 'DT', 'pct_cond', 'dpr_val', 'note', 'asd_val']
fixtures.csv: ['acct', 'bld_num', 'type', 'type_dscr', 'units']
land.csv: ['acct', 'num', 'use_cd', 'use_dscr', 'inf_cd', 'inf_dscr', 'inf_adj', 'tp', 'uts', 'sz_fact', 'inf_fact', 'cond', 'ovr_dscr', 'tot_adj', 'unit_prc', 'adj_unit_prc', 'val', 'ovr_val']
land_ag.csv: ['acct', 'num', 'use_cd', 'use_dscr', 'inf_cd', 'in

In [10]:
import os
print("All folders in houston_data:")
for item in os.listdir("Houston_data"):
    path = f"Houston_data/{item}"
    if os.path.isdir(path):
        print(f"  📁 {item}/")
        # Count files inside
        file_count = len([f for f in os.listdir(path) if f.endswith('.csv')])
        print(f"     ({file_count} CSV files)")

All folders in houston_data:
  📁 csv_files/
     (0 CSV files)
  📁 txt_files/
     (0 CSV files)


In [12]:
import pandas as pd
import os

print("Searching for address columns across all HCAD CSVs...\n")

csv_dir = "Houston_data/csv_files"
address_keywords = ["address", "addr", "street", "st_", "road", "ave", "blvd", "city", "zip", "postal", "house", "number", "name"]

found_address = False

for folder in os.listdir(csv_dir):
    folder_path = os.path.join(csv_dir, folder)
    if os.path.isdir(folder_path):
        for csv_file in os.listdir(folder_path):
            if csv_file.endswith(".csv"):
                try:
                    df = pd.read_csv(os.path.join(folder_path, csv_file), nrows=1)
                    cols = [c.lower() for c in df.columns]
                    
                    # Check if any address keywords match
                    matches = [c for c in cols if any(kw in c for kw in address_keywords)]
                    
                    if matches:
                        found_address = True
                        print(f"✓ {folder}/{csv_file}")
                        print(f"  Found: {matches}")
                        print()
                except:
                    pass

if not found_address:
    print("❌ No address columns found in any CSV files.")
    print("\nHowever, let's check what's IN Real_acct_owner/ folder:")
    print("(We may have owner/account info that includes addresses)\n")
    
    for csv_file in os.listdir("Houston_data/csv_files/Real_acct_owner"):
        if csv_file.endswith(".csv"):
            try:
                df = pd.read_csv(f"Houston_data/csv_files/Real_acct_owner/{csv_file}", nrows=1)
                print(f"{csv_file}: {df.columns.tolist()}")
            except:
                pass

Searching for address columns across all HCAD CSVs...

✓ Real_acct_owner/real_mnrl.csv
  Found: ['interest_percent']

✓ Real_jur_exempt/jur_tax_dist_exempt_value_rate.csv
  Found: ['name']

